# Zehirli / Zehirsiz Böcek Sınıflandırması — Model Eğitimi

Bu notebook, iNaturalist'ten toplanan iki veri setini (*Zehirli Bocek Veri Toplama* ve *Zehirli Bocek Ek Veri*) birleştirip iki farklı yaklaşımı eğitir ve karşılaştırır:

| | Yaklaşım | Çıktı |
|---|---|---|
| **Model A** | Doğrudan ikili sınıflandırma | zehirli / zehirsiz |
| **Model B** | Hiyerarşik: önce 38 takson grubu, sonra gruptan zehir durumu | grup → zehirli / zehirsiz |

**Model:** ImageNet üzerinde önceden eğitilmiş **EfficientNet-B0** (transfer öğrenme), 224×224 giriş.

**Veri:** 38 gruptan her birinden 190 fotoğraf (toplam ~7.200) doğrudan iNaturalist'ten çekilip 224×224 olarak belleğe alınır; veri artırma (yatay çevirme, rastgele kırpma/yakınlaştırma, parlaklık ve kontrast) GPU üzerinde yapılır.

**Veri bölme:**
- Her gruptaki türlerin bir kısmı **hiç eğitime girmez** (*görülmemiş tür testi*): modelin daha önce hiç görmediği türlerde zehirliliği ne kadar genelleyebildiğini ölçer.
- Kalan veri gruplara göre katmanlı olarak %80 eğitim, %10 doğrulama, %10 test şeklinde bölünür.

**Dengesizlik:** Ağırlıklı örnekleme: her sınıf (zehirli/zehirsiz) %50, sınıf içinde her grup eşit ağırlıkla örneklenir (oversampling).

**Değerlendirme:** Accuracy, F1, precision, zehirliler için recall (kaçırılan zehirli böcek tehlikelidir), ROC-AUC, karışıklık matrisleri, gruplara göre doğruluk, benzer görünümlü çiftler analizi ve Grad-CAM.

## 1. Kütüphaneler ve ayarlar

In [ ]:
import os, glob, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, confusion_matrix)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda"
EPOCH = 10
BATCH = 128
LR = 3e-4
IMG = 224
GRUP_BASINA = 190
CIKTI = "/kaggle/working"
print("GPU sayısı:", torch.cuda.device_count(), "| CPU:", os.cpu_count())

## 2. Fotoğrafları iNaturalist'ten çekme (her gruptan 190, toplam ~7.200)

Önceki notebook'larda (*Zehirli Bocek Veri Toplama* ve *Ek Veri*) tanımlanan 38 grubun her birinden 190 fotoğraf doğrudan iNaturalist'ten çekilir: Research Grade gözlemler, CC0 / CC-BY / CC-BY-NC lisanslı fotoğraflar, tırtıl gruplarında yalnızca larva evresi. Fotoğraflar 224×224'e getirilip doğrudan belleğe alınır.

In [ ]:
import io, requests

API = "https://api.inaturalist.org/v1"
LISANSLAR = ["cc0", "cc-by", "cc-by-nc"]
LARVA = {"term_id": 1, "term_value_id": 6}

GRUPLAR_TANIM = [
    # takson,          sınıf,      larva, hariç tutulan
    ("Vespidae", "zehirli", False, None), ("Apis mellifera", "zehirli", False, None),
    ("Bombus", "zehirli", False, None), ("Solenopsis", "zehirli", False, None),
    ("Pompilidae", "zehirli", False, None), ("Mutillidae", "zehirli", False, None),
    ("Meloidae", "zehirli", False, None), ("Paederus", "zehirli", False, None),
    ("Reduviidae", "zehirli", False, None), ("Thaumetopoea", "zehirli", True, None),
    ("Megalopygidae", "zehirli", True, None), ("Limacodidae", "zehirli", True, None),
    ("Pogonomyrmex", "zehirli", False, None), ("Hemileucinae", "zehirli", True, None),
    ("Lymantriinae", "zehirli", True, None),
    ("Syrphidae", "zehirsiz", False, None), ("Muscidae", "zehirsiz", False, None),
    ("Tipulidae", "zehirsiz", False, None), ("Pieridae", "zehirsiz", False, None),
    ("Lycaenidae", "zehirsiz", False, None), ("Odonata", "zehirsiz", False, None),
    ("Acrididae", "zehirsiz", False, None), ("Tettigoniidae", "zehirsiz", False, None),
    ("Mantodea", "zehirsiz", False, None), ("Cicadidae", "zehirsiz", False, None),
    ("Cerambycidae", "zehirsiz", False, None), ("Chrysopidae", "zehirsiz", False, None),
    ("Meliponini", "zehirsiz", False, None), ("Formicinae", "zehirsiz", False, None),
    ("Staphylinidae", "zehirsiz", False, "Paederus"), ("Notodontidae", "zehirsiz", True, "Thaumetopoeinae"),
    ("Saturniinae", "zehirsiz", True, None), ("Sphingidae", "zehirsiz", True, None),
    ("Tenthredinidae", "zehirsiz", False, None), ("Sesiidae", "zehirsiz", False, None),
    ("Pentatomidae", "zehirsiz", False, None), ("Coreidae", "zehirsiz", False, None),
    ("Chrysomelidae", "zehirsiz", False, None),
]

oturum = requests.Session()
oturum.headers["User-Agent"] = "zehirli-bocek-ders-projesi (Kaggle notebook)"

def api(yol, **params):
    for deneme in range(5):
        time.sleep(1.0)
        try:
            r = oturum.get(f"{API}/{yol}", params=params, timeout=60)
            if r.status_code == 200:
                return r.json()
        except requests.RequestException:
            pass
        time.sleep(5 * (deneme + 1))
    raise RuntimeError(f"API isteği başarısız: {yol}")

def takson_id(ad):
    return next(t["id"] for t in api("taxa", q=ad, is_active="true", per_page=30)["results"] if t["name"] == ad)

satirlar = []
t0 = time.time()
for ad, sinif, larva, haric in GRUPLAR_TANIM:
    p = dict(taxon_id=takson_id(ad), quality_grade="research", photos="true",
             photo_license=",".join(LISANSLAR), per_page=200, order_by="id", order="desc")
    if haric:
        p["without_taxon_id"] = takson_id(haric)
    if larva:
        p.update(LARVA)
    n, sayfa = 0, 1
    while n < GRUP_BASINA and sayfa <= 5:
        sonuc = api("observations", page=sayfa, **p)["results"]
        if not sonuc:
            break
        for o in sonuc:
            foto = next((f for f in o["photos"] if f.get("license_code") in LISANSLAR), None)
            if foto is None or n >= GRUP_BASINA:
                continue
            satirlar.append(dict(sinif=sinif, grup=ad.replace(" ", "_"), photo_id=foto["id"],
                                 takson=o["taxon"]["name"], lisans=foto["license_code"], atif=foto.get("attribution"),
                                 url=foto["url"].replace("/square.", "/medium.")))
            n += 1
        sayfa += 1
    print(f"{ad:16s} {sinif:9s} {n} fotoğraf", flush=True)

veri = pd.DataFrame(satirlar).drop_duplicates("photo_id").reset_index(drop=True)
print(f"Liste hazır: {len(veri)} fotoğraf ({(time.time() - t0) / 60:.1f} dk)")

In [ ]:
GORUNTU = np.zeros((len(veri), IMG, IMG, 3), dtype=np.uint8)
basarili = np.zeros(len(veri), dtype=bool)

def indir(i):
    for deneme in range(3):
        try:
            r = requests.get(veri.url.iat[i], timeout=30)
            if r.status_code == 200:
                img = Image.open(io.BytesIO(r.content)).convert("RGB")
                k = IMG / min(img.size)
                img = img.resize((max(IMG, round(img.width * k)), max(IMG, round(img.height * k))), Image.BILINEAR)
                sol, ust = (img.width - IMG) // 2, (img.height - IMG) // 2
                GORUNTU[i] = np.asarray(img.crop((sol, ust, sol + IMG, ust + IMG)))
                basarili[i] = True
                return
        except Exception:
            pass
        time.sleep(2)

t0 = time.time()
with ThreadPoolExecutor(16) as ex:
    for j, _ in enumerate(ex.map(indir, range(len(veri))), 1):
        if j % 2000 == 0:
            print(f"{j}/{len(veri)} indirildi ({(time.time() - t0) / 60:.1f} dk)", flush=True)

GORUNTU = GORUNTU[basarili]
veri = veri[basarili].reset_index(drop=True)
GORUNTU_T = torch.from_numpy(GORUNTU)
veri.drop(columns="url").to_csv(f"{CIKTI}/metadata.csv", index=False)

veri["idx"] = np.arange(len(veri))
veri["etiket"] = (veri.sinif == "zehirli").astype(int)
GRUPLAR = sorted(veri.grup.unique())
grup_idx = {g: i for i, g in enumerate(GRUPLAR)}
veri["grup_id"] = veri.grup.map(grup_idx)
grup_sinif = veri.groupby("grup").sinif.first()
ZEHIRLI_GRUP_IDS = [grup_idx[g] for g in GRUPLAR if grup_sinif[g] == "zehirli"]

print(f"İndirme bitti: {len(veri)} fotoğraf | {(time.time() - t0) / 60:.1f} dk | {GORUNTU.nbytes / 1e9:.2f} GB")
print(f"{len(GRUPLAR)} grup | {veri.takson.nunique()} takson |", veri.sinif.value_counts().to_dict())

## 3. Eğitim / doğrulama / test ve görülmemiş tür bölmesi

In [ ]:
rng = np.random.default_rng(SEED)

# Görülmemiş tür testi: her grupta (en az 10 farklı taksonu olan) grubun ~%10'u kadar fotoğrafı
# oluşturan bazı taksonlar tamamen ayrılır. Tek başına grubun %5'inden büyük taksonlar ayrılmaz.
gorulmemis = []
for g, d in veri.groupby("grup"):
    say = d.takson.value_counts()
    if len(say) < 10:
        continue
    taksonlar = list(say.index)
    rng.shuffle(taksonlar)
    hedef, toplam = 0.10 * len(d), 0
    for t in taksonlar:
        if toplam >= hedef:
            break
        if say[t] > 0.05 * len(d):
            continue
        gorulmemis.append(t)
        toplam += say[t]

veri["bolum"] = "egitim"
veri.loc[veri.takson.isin(gorulmemis), "bolum"] = "gorulmemis_tur"

# Kalan veri: her grup içinde %80 eğitim, %10 doğrulama, %10 test
for g, idx in veri[veri.bolum == "egitim"].groupby("grup").groups.items():
    idx = rng.permutation(np.array(idx))
    n = len(idx)
    veri.loc[idx[: int(0.1 * n)], "bolum"] = "dogrulama"
    veri.loc[idx[int(0.1 * n): int(0.2 * n)], "bolum"] = "test"

veri.to_csv(f"{CIKTI}/veri_bolmesi.csv", index=False)
bol = {b: veri[veri.bolum == b].reset_index(drop=True) for b in ["egitim", "dogrulama", "test", "gorulmemis_tur"]}
print("Görülmemiş takson sayısı:", len(gorulmemis))
pd.crosstab(veri.bolum, veri.sinif, margins=True)

## 4. Veri artırma (GPU üzerinde) ve örnekleme

In [ ]:
ORT = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)

def batch_hazirla(idx, egitim):
    """Bellekteki fotoğraflardan bir batch oluşturur; eğitimde GPU üzerinde veri artırma uygular."""
    x = GORUNTU_T[torch.as_tensor(idx)].to(DEVICE).permute(0, 3, 1, 2).contiguous().float().div_(255)
    if egitim:
        cevir = torch.rand(len(x), device=DEVICE) < 0.5                 # yatay çevirme
        x[cevir] = x[cevir].flip(3)
        s = int(IMG * np.random.uniform(0.7, 1.0))                       # rastgele kırpma + yakınlaştırma
        i0, j0 = np.random.randint(0, IMG - s + 1, 2)
        x = F.interpolate(x[:, :, i0:i0 + s, j0:j0 + s], size=IMG, mode="bilinear", align_corners=False)
        parlaklik = 1 + (torch.rand(len(x), 1, 1, 1, device=DEVICE) - 0.5) * 0.4
        kontrast = 1 + (torch.rand(len(x), 1, 1, 1, device=DEVICE) - 0.5) * 0.4
        ort = x.mean(dim=(1, 2, 3), keepdim=True)
        x = (((x - ort) * kontrast + ort) * parlaklik).clamp_(0, 1)     # renk tonu değiştirilmez: uyarı renkleri önemli
    return (x - ORT) / STD

tr = bol["egitim"]
TR_IDX, TR_E, TR_G = tr.idx.values, torch.tensor(tr.etiket.values), torch.tensor(tr.grup_id.values)

# Her sınıf %50, sınıf içinde her grup eşit olasılıkla örneklenir
grup_sayisi = tr.groupby("sinif").grup.nunique()
AGIRLIK = torch.tensor((1.0 / (tr.sinif.map(grup_sayisi) * tr.grup.map(tr.grup.value_counts()))).values)
ADIM = len(tr) // BATCH

def egitim_batchleri():
    secim = torch.multinomial(AGIRLIK, ADIM * BATCH, replacement=True).view(ADIM, BATCH)
    for s in secim:
        yield TR_IDX[s.numpy()], TR_E[s], TR_G[s]

def eval_batchleri(df, boyut=256):
    for i in range(0, len(df), boyut):
        yield df.idx.values[i:i + boyut]

print({b: len(d) for b, d in bol.items()}, "| epoch başına adım:", ADIM)

In [ ]:
# Veri artırma örnekleri
idx, e, g = next(egitim_batchleri())
x = (batch_hazirla(idx[:16], True) * STD + ORT).clamp(0, 1).permute(0, 2, 3, 1).cpu().numpy()
fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for ax, xi, ei, gi in zip(axes.flat, x, e[:16], g[:16]):
    ax.imshow(xi)
    ax.set_title(f"{GRUPLAR[gi]}\n{'zehirli' if ei else 'zehirsiz'}", fontsize=8, color="red" if ei else "green")
    ax.axis("off")
plt.suptitle("Eğitim örnekleri (veri artırma uygulanmış)"); plt.tight_layout(); plt.show()

## 5. Model ve eğitim fonksiyonları

In [ ]:
def model_olustur(n_cikis):
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_cikis)
    return m

@torch.no_grad()
def tahmin_et(model, df, tip):
    """P(zehirli) ve (Model B için) grup olasılıklarını döndürür."""
    model.eval()
    p_zehirli, p_grup = [], []
    for idx in eval_batchleri(df):
        with torch.autocast("cuda", dtype=torch.float16):
            x = batch_hazirla(idx, False)
            # TTA: normal ve yatay aynalanmış görüntünün olasılık ortalaması
            prob = (model(x).float().softmax(1) + model(x.flip(3)).float().softmax(1)) / 2
        if tip == "A":
            p_zehirli.append(prob[:, 1].cpu())
        else:
            p_grup.append(prob.cpu())
            p_zehirli.append(prob[:, ZEHIRLI_GRUP_IDS].sum(1).cpu())
    return torch.cat(p_zehirli).numpy(), (torch.cat(p_grup).numpy() if p_grup else None)

def egit(tip):
    n_cikis = 2 if tip == "A" else len(GRUPLAR)
    model = model_olustur(n_cikis).to(DEVICE)   # tek GPU yeterli; DataParallel kullanılmaz
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCH * ADIM, pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda")
    kayip_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
    y_val = bol["dogrulama"].etiket.values
    gecmis = []
    for ep in range(1, EPOCH + 1):
        model.train()
        t0, toplam, n = time.time(), 0.0, 0
        for i, (idx, e, g) in enumerate(egitim_batchleri(), 1):
            x = batch_hazirla(idx, True)
            y = (e if tip == "A" else g).to(DEVICE)
            with torch.autocast("cuda", dtype=torch.float16):
                kayip = kayip_fn(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(kayip).backward()
            scaler.step(opt)
            scaler.update()
            sch.step()
            toplam += kayip.item() * len(x)
            n += len(x)
            if i == 1 or i % 50 == 0:
                print(f"  [Model {tip}] epoch {ep} adım {i}/{ADIM} | {n / (time.time() - t0):.0f} foto/sn", flush=True)
        p, _ = tahmin_et(model, bol["dogrulama"], tip)
        f1, rec = f1_score(y_val, p > 0.5), recall_score(y_val, p > 0.5)
        gecmis.append(dict(epoch=ep, kayip=toplam / n, val_f1=f1, val_recall=rec))
        print(f"[Model {tip}] epoch {ep}/{EPOCH} | kayıp={toplam / n:.4f} | val F1={f1:.4f} | "
              f"val recall(zehirli)={rec:.4f} | {(time.time() - t0) / 60:.1f} dk", flush=True)
    torch.save((model.module if hasattr(model, "module") else model).state_dict(), f"{CIKTI}/model_{tip}.pt")
    return model, pd.DataFrame(gecmis)

## 6. Model A — doğrudan zehirli / zehirsiz

In [ ]:
model_A, gecmis_A = egit("A")

## 7. Model B — önce takson grubu, sonra zehir durumu

In [ ]:
model_B, gecmis_B = egit("B")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for tip, g in [("A", gecmis_A), ("B", gecmis_B)]:
    ax[0].plot(g.epoch, g.kayip, marker="o", label=f"Model {tip}")
    ax[1].plot(g.epoch, g.val_f1, marker="o", label=f"Model {tip}")
ax[0].set_title("Eğitim kaybı"); ax[1].set_title("Doğrulama F1 (zehirli)")
for a in ax:
    a.set_xlabel("Epoch"); a.legend(); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8. Sonuçlar: test seti ve görülmemiş türler

In [ ]:
sonuclar, tahminler = [], {}
for tip, model in [("A", model_A), ("B", model_B)]:
    for b in ["test", "gorulmemis_tur"]:
        p, pg = tahmin_et(model, bol[b], tip)
        y, t = bol[b].etiket.values, (p > 0.5).astype(int)
        tahminler[(tip, b)] = (p, pg)
        sonuclar.append(dict(
            model=tip, bolum=b, n=len(y),
            accuracy=accuracy_score(y, t), f1=f1_score(y, t),
            precision=precision_score(y, t), recall_zehirli=recall_score(y, t),
            roc_auc=roc_auc_score(y, p),
            grup_dogrulugu=(pg.argmax(1) == bol[b].grup_id.values).mean() if pg is not None else np.nan,
        ))
sonuc_df = pd.DataFrame(sonuclar)
sonuc_df.to_csv(f"{CIKTI}/sonuclar.csv", index=False)
sonuc_df.round(4)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, ((tip, b), (p, _)) in zip(axes.flat, tahminler.items()):
    cm = confusion_matrix(bol[b].etiket.values, (p > 0.5).astype(int))
    sns.heatmap(cm, annot=True, fmt="d", cbar=False, ax=ax, cmap="Reds" if tip == "A" else "Blues",
                xticklabels=["zehirsiz", "zehirli"], yticklabels=["zehirsiz", "zehirli"])
    ax.set_title(f"Model {tip} — {b}"); ax.set_xlabel("Tahmin"); ax.set_ylabel("Gerçek")
plt.tight_layout(); plt.show()

## 9. Gruplara göre doğruluk

In [ ]:
d = bol["test"][["grup", "sinif"]].copy()
for tip in ["A", "B"]:
    d[f"dogru_{tip}"] = (tahminler[(tip, "test")][0] > 0.5).astype(int) == bol["test"].etiket.values
grup_basari = d.groupby(["sinif", "grup"])[["dogru_A", "dogru_B"]].mean().sort_values("dogru_A")
grup_basari.to_csv(f"{CIKTI}/grup_basarisi.csv")

ax = grup_basari.plot.barh(figsize=(10, 13), color=["tab:red", "tab:blue"])
ax.set_xlabel("Zehirli / zehirsiz doğruluğu (test)"); ax.set_title("Gruplara göre doğruluk")
ax.legend(["Model A", "Model B"]); plt.tight_layout(); plt.show()
grup_basari.round(3)

## 10. Benzer görünümlü çiftler

Aynı aile veya benzer görünümdeki zehirli–zehirsiz çiftlerinde:
- Model A'nın her iki gruptaki ikili doğruluğu
- Model B'nin bu iki grubu birbiriyle ne sıklıkla karıştırdığı

In [ ]:
CIFTLER = [
    ("Apis_mellifera", "Meliponini"), ("Bombus", "Meliponini"),
    ("Vespidae", "Syrphidae"), ("Vespidae", "Tenthredinidae"), ("Vespidae", "Sesiidae"),
    ("Solenopsis", "Formicinae"), ("Pogonomyrmex", "Formicinae"),
    ("Paederus", "Staphylinidae"), ("Thaumetopoea", "Notodontidae"),
    ("Hemileucinae", "Saturniinae"), ("Megalopygidae", "Sphingidae"),
    ("Limacodidae", "Sphingidae"), ("Lymantriinae", "Sphingidae"),
    ("Reduviidae", "Pentatomidae"), ("Reduviidae", "Coreidae"),
    ("Meloidae", "Chrysomelidae"),
]
pg = tahminler[("B", "test")][1].argmax(1)
gercek = bol["test"].grup.values
satirlar = []
for z, s in CIFTLER:
    satirlar.append(dict(
        zehirli=z, zehirsiz_benzeri=s,
        A_zehirli_dogru=grup_basari.loc[("zehirli", z), "dogru_A"],
        A_zehirsiz_dogru=grup_basari.loc[("zehirsiz", s), "dogru_A"],
        B_zehirliyi_zehirsiz_sandi=(pg[gercek == z] == grup_idx[s]).mean(),
        B_zehirsizi_zehirli_sandi=(pg[gercek == s] == grup_idx[z]).mean(),
    ))
cift_df = pd.DataFrame(satirlar)
cift_df.to_csv(f"{CIKTI}/benzer_ciftler.csv", index=False)
cift_df.round(3)

## 11. Grad-CAM: Model A fotoğrafın neresine bakıyor?

In [ ]:
def grad_cam(model, x, hedef=1):
    m = model.module if hasattr(model, "module") else model
    m.eval()
    kayit = {}
    katman = m.features[-1]
    h1 = katman.register_forward_hook(lambda mod, i, o: kayit.__setitem__("akt", o))
    h2 = katman.register_full_backward_hook(lambda mod, gi, go: kayit.__setitem__("grad", go[0]))
    out = m(x)
    m.zero_grad()
    out[0, hedef].backward()
    h1.remove(); h2.remove()
    w = kayit["grad"].mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * kayit["akt"]).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=(IMG, IMG), mode="bilinear")[0, 0].detach()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam.cpu().numpy(), out.softmax(1)[0, 1].item()

ornek_gruplar = ["Apis_mellifera", "Meliponini", "Vespidae", "Syrphidae", "Paederus",
                 "Staphylinidae", "Thaumetopoea", "Notodontidae", "Hemileucinae", "Saturniinae"]
ornek = bol["test"].groupby("grup").head(1).set_index("grup").loc[ornek_gruplar].reset_index()
fig, axes = plt.subplots(2, len(ornek), figsize=(2.5 * len(ornek), 6))
for j, s in enumerate(ornek.itertuples()):
    cam, p = grad_cam(model_A, batch_hazirla([s.idx], False))
    axes[0, j].imshow(GORUNTU[s.idx])
    axes[0, j].set_title(f"{s.grup}\n({s.sinif})", fontsize=8, color="red" if s.sinif == "zehirli" else "green")
    axes[1, j].imshow(GORUNTU[s.idx]); axes[1, j].imshow(cam, cmap="jet", alpha=0.45)
    axes[1, j].set_title(f"P(zehirli) = {p:.2f}", fontsize=8)
    for a in axes[:, j]:
        a.axis("off")
plt.suptitle("Grad-CAM — üst: fotoğraf, alt: modelin dikkat ettiği bölgeler")
plt.tight_layout(); plt.show()

## 12. Eşik analizi ve A+B birleşimi (ensemble)

Varsayılan karar eşiği 0,5'tir. Zehirli bir böceği kaçırmak daha tehlikeli olduğu için eşik **doğrulama setinde** seçilir (test ve görülmemiş türler bu seçimde kullanılmaz):
- **F1-en iyi eşik:** doğrulama F1'ini en yüksek yapan eşik
- **Güvenli eşik:** zehirlileri en az %90 yakalayan en yüksek eşik

Ayrıca Model A ve Model B'nin olasılıklarının ortalaması (ensemble) da değerlendirilir.

In [ ]:
def olc(y, p, esik):
    t = (p >= esik).astype(int)
    return dict(accuracy=accuracy_score(y, t), f1=f1_score(y, t), precision=precision_score(y, t, zero_division=0),
                recall_zehirli=recall_score(y, t), roc_auc=roc_auc_score(y, p))

olasilik = {}
for b in ["dogrulama", "test", "gorulmemis_tur"]:
    pA, _ = tahmin_et(model_A, bol[b], "A")
    pB, _ = tahmin_et(model_B, bol[b], "B")
    olasilik[b] = {"A": pA, "B": pB, "A+B": (pA + pB) / 2}

esikler = np.arange(0.05, 0.96, 0.01)
y_val = bol["dogrulama"].etiket.values
satir = []
for m in ["A", "B", "A+B"]:
    pv = olasilik["dogrulama"][m]
    f1_esik = esikler[np.argmax([f1_score(y_val, pv >= e) for e in esikler])]
    guvenli = [e for e in esikler if recall_score(y_val, pv >= e) >= 0.90]
    guvenli_esik = max(guvenli) if guvenli else esikler[0]
    for ad, e in [("0.50", 0.5), ("F1-en iyi", f1_esik), ("güvenli (recall≥%90)", guvenli_esik)]:
        for b in ["test", "gorulmemis_tur"]:
            satir.append(dict(model=m, esik_tipi=ad, esik=round(float(e), 2), bolum=b,
                              **olc(bol[b].etiket.values, olasilik[b][m], e)))
esik_df = pd.DataFrame(satir)
esik_df.to_csv(f"{CIKTI}/esik_analizi.csv", index=False)
esik_df.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, b in zip(axes, ["test", "gorulmemis_tur"]):
    y = bol[b].etiket.values
    for m in ["A", "B", "A+B"]:
        p = olasilik[b][m]
        ax.plot(esikler, [f1_score(y, p >= e) for e in esikler], label=f"{m} — F1")
        ax.plot(esikler, [recall_score(y, p >= e) for e in esikler], "--", label=f"{m} — recall")
    ax.set_title(f"Eşiğe göre F1 ve zehirli yakalama — {b}"); ax.set_xlabel("Eşik"); ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 13. Klasik makine öğrenmesi karşılaştırması: Random Forest, SVM, Lojistik Regresyon

Eğitilmiş Model B'nin son katmanından önceki **1.280 boyutlu özellik vektörü** her fotoğraf için çıkarılır. Bu özellikler üzerinde klasik sınıflandırıcılar (zehirli / zehirsiz) eğitilir ve derin öğrenme modelleriyle karşılaştırılır. (Ham pikseller üzerinde Random Forest çok başarısız olacağı için bu hibrit yöntem kullanılır.)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

@torch.no_grad()
def ozellik_cikar(model, df):
    model.eval()
    govde = nn.Sequential(model.features, model.avgpool, nn.Flatten())
    cikti = []
    for idx in eval_batchleri(df):
        with torch.autocast("cuda", dtype=torch.float16):
            x = batch_hazirla(idx, False)
            cikti.append(((govde(x) + govde(x.flip(3))) / 2).float().cpu())
    return torch.cat(cikti).numpy()

t0 = time.time()
OZ = {b: ozellik_cikar(model_B, bol[b]) for b in ["egitim", "test", "gorulmemis_tur"]}
print(f"Özellikler çıkarıldı: {OZ['egitim'].shape} ({(time.time() - t0) / 60:.1f} dk)")

siniflandiricilar = {
    "Random Forest": RandomForestClassifier(n_estimators=400, class_weight="balanced", n_jobs=-1, random_state=SEED),
    "SVM (doğrusal)": make_pipeline(StandardScaler(), CalibratedClassifierCV(LinearSVC(C=0.05, class_weight="balanced", max_iter=5000), cv=3)),
    "Lojistik Regresyon": make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=3000, class_weight="balanced")),
}
y_tr = bol["egitim"].etiket.values
klasik = []
for ad, clf in siniflandiricilar.items():
    t0 = time.time()
    clf.fit(OZ["egitim"], y_tr)
    for b in ["test", "gorulmemis_tur"]:
        p = clf.predict_proba(OZ[b])[:, 1]
        klasik.append(dict(model=ad, bolum=b, **olc(bol[b].etiket.values, p, 0.5)))
    print(f"{ad}: {(time.time() - t0):.0f} sn", flush=True)
for m in ["A", "B", "A+B"]:
    for b in ["test", "gorulmemis_tur"]:
        klasik.append(dict(model=f"CNN {m} (TTA)", bolum=b, **olc(bol[b].etiket.values, olasilik[b][m], 0.5)))
klasik_df = pd.DataFrame(klasik).sort_values(["bolum", "f1"], ascending=[True, False])
klasik_df.to_csv(f"{CIKTI}/klasik_vs_derin.csv", index=False)
klasik_df.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, b in zip(axes, ["test", "gorulmemis_tur"]):
    d = klasik_df[klasik_df.bolum == b].set_index("model")[["accuracy", "f1", "recall_zehirli"]]
    d.plot.barh(ax=ax)
    ax.set_title(f"Klasik ML ve derin öğrenme — {b}"); ax.set_xlim(0, 1); ax.grid(alpha=0.3, axis="x")
plt.tight_layout(); plt.show()

## 14. Özet

In [ ]:
print("=" * 70)
print("ZEHİRLİ / ZEHİRSİZ BÖCEK SINIFLANDIRMASI — SONUÇ ÖZETİ")
print("=" * 70)
for r in sonuc_df.itertuples():
    print(f"Model {r.model} | {r.bolum:15s} | acc={r.accuracy:.3f} | F1={r.f1:.3f} | "
          f"recall(zehirli)={r.recall_zehirli:.3f} | AUC={r.roc_auc:.3f}")
print("\nEn iyi klasik/derin (görülmemiş türler, F1):")
print(klasik_df[klasik_df.bolum == "gorulmemis_tur"][["model", "accuracy", "f1", "recall_zehirli"]].round(3).to_string(index=False))
print("\nKaydedilen dosyalar:", sorted(f for f in os.listdir(CIKTI) if not f.startswith("__")))